##### Library imports

In [ ]:
import re, os, sys
import pandas as pd
import cv2 as cv
import SimpleITK as sitk
import numpy as np
from skimage import filters
import json
import matplotlib.pyplot as plt

##### Function definitions

In [ ]:
def rotate_image(image, dimension = 3):
    """Rotates NIfTI axial images so they are upright for visualization. Simple insights tool kit (SITK) utilizes the LPS coordinate system, 
       meaning image voxel coordinates are assumed to increase in the Right --> Left, Anterior --> Posterior, and Inferior --> Superior directions. 
       However, NIfTI images use the RAS coordinate system, which makes axial slices (lateral cross-sections) appear inverted when read with SITK. 
       This function resamples the image such that it is visualized in upright direction, enabling interpretable visualization and processing. 
    
    Parameters
    ----------
    image : sitk.Image
        The original NIfTI image (Note that the direction cosine should be [-1,0,0,0,-1,0,0,0,1])
    dimension : int, optional
        Dimensionality of the original and rotation-corrected image. Default is 3.
    
    Returns
    -------
    rotated_image : sitk.Image
        Rotation-corrected image, with a direction cosine of [1,0,0,0,1,0,0,0,1]
    """

    transform = sitk.AffineTransform(dimension)
    transform.SetCenter(image.TransformContinuousIndexToPhysicalPoint(np.array(image.GetSize())//2.0))
    s = image.GetSpacing()[2]

    matrix = np.array([[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,1.0]])

   
    transform.SetMatrix(matrix.ravel())

    extreme_points = [image.TransformIndexToPhysicalPoint((0,0,0)), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,0)),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),0)),
                      image.TransformIndexToPhysicalPoint((0,0,image.GetDepth())), 
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),0,image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((image.GetWidth(),image.GetHeight(),image.GetDepth())),
                      image.TransformIndexToPhysicalPoint((0,image.GetHeight(),image.GetDepth()))]

    inv_transform = transform.GetInverse()

    extreme_points_transformed = [inv_transform.TransformPoint(pnt) for pnt in extreme_points]
    min_x = min(extreme_points_transformed)[0]
    min_y = min(extreme_points_transformed, key=lambda p: p[1])[1]
    min_z = min(extreme_points_transformed, key=lambda p: p[2])[2]
    max_x = max(extreme_points_transformed)[0]
    max_y = max(extreme_points_transformed, key=lambda p: p[1])[1]
    max_z = max(extreme_points_transformed, key=lambda p: p[2])[2]

    # Use the original spacing (arbitrary decision).
    output_spacing = image.GetSpacing()
    # Identity cosine matrix.   
    output_direction = [1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0]
    # Minimal x,y coordinates are the new origin.
    output_origin = [min_x, min_y, min_z]
    
    # Compute grid size based on the physical size and spacing.
    output_size = image.GetSize()
    
    rotated_image = sitk.Resample(image, output_size, transform, sitk.sitkLinear, output_origin, output_spacing,
                                  output_direction)
    return rotated_image

In [ ]:
def window_stack_sitk(input_im, window_center=40, window_width=80):
    """
    Clamps values outside [min, max] to the edge values.
    
    Parameters
    ----------
    input_im : sitk.Image
        Input, unwindowed, brain image
    window_center : int, optional
        The center of the Hounsfield Unit (HU) scale in the windowed image. Default is 40 HU (brain).
    window_width : int, optional
        The total HU window width around the window_center, in the windowed image. Default is 80 HU (brain).

    Returns
    -------
    windowed_im : sitk.Image
        Windowed CT image with the desired tissue's visualization optimized (e.g., 0 - 80 HU for brain tissue)
    
    """
    img_min = window_center - (window_width // 2)
    img_max = window_center + (window_width // 2)


    windowed_im = sitk.IntensityWindowing(
        input_im, 
        windowMinimum=float(img_min), 
        windowMaximum=float(img_max), 
        outputMinimum=float(img_min), 
        outputMaximum=float(img_max)
    )
    
    return windowed_im

In [ ]:
def getLargestCC(blobs_labels):
    """Returns the largest connected component from an image containing blobs of discrete labels
    
    Parameters
    ----------
    blobs_labels : numpy.ndarray 
        A collection of discrete blobs (e.g., cerebrospinal fluid [CSF] spaces including the lateral ventricles, longitudinal fissure, sulci etc.)

    Returns
    -------
    largestCC: numpy.ndarray
        The largest connected component from a collection of discrete blobs (e.g. lateral ventricles from a collection of CSF spaces)  
    
    """
    # assume at least 1 CC apart from background
    if blobs_labels.max() == 0:
        raise ValueError('Blank segmentation, inspect processing up to here.')
    #this assumes that the background is the largest CC (label 0)
    largestCC = blobs_labels == np.argmax(np.bincount(blobs_labels.flat)[1:])+1 
    return largestCC

In [ ]:
def ventricle_fullbrain_seg(ax_img_brain_arr, visualize=True):
    """
    Segments the ventricles and the full brain mask from a 3D axial image volume 
    using Otsu thresholding and background flood-filling.
    
    Parameters
    ----------
    ax_img_brain_arr : numpy.ndarray
        A 3D array representing the axial brain image volume.
    visualize : bool, optional
        If True, displays 2D sum projections (IS, AP, LR) of the segmented 
        ventricles and the full brain mask. Default is True.

    Returns
    -------
    ventricles : numpy.ndarray
        A 3D binary mask of the segmented ventricles, same shape as the input array.
    full_brain : numpy.ndarray
        A 3D binary mask of the full brain, same shape as the input array.
    """
    
    ventricles = np.zeros(ax_img_brain_arr.shape)
    full_brain = np.zeros(ax_img_brain_arr.shape)
    
    for pl in range(len(ax_img_brain_arr)):
        ax_plane = ax_img_brain_arr[pl,:,:]

        try:
            thresh = filters.threshold_otsu(ax_plane)
        except ValueError:
            continue
            
        ax_bin = ax_plane > thresh

        # Floodfilling to extract the full brain mask
        im_th = ax_bin.copy()
        h, w = im_th.shape[:2]
        mask = np.zeros((h+2, w+2), np.uint8)
        im_floodfill = im_th.copy()
        # Floodfill from point (0, 0)
        res = cv.floodFill(np.uint8(im_floodfill), mask, (0,0), 255)

        full_brain_mask = (1-res[2]).copy()
        full_brain_mask = full_brain_mask[1:h+1,1:w+1]

        ventricles[pl,:,:] = full_brain_mask - ax_bin
        full_brain[pl,:,:] = full_brain_mask
    
    if visualize:
        ventricle_is = np.sum(ventricles, axis = 0)
        ventricle_ap = np.sum(ventricles, axis = 1)
        ventricle_lr = np.sum(ventricles, axis = 2)
        fig,(ax1, ax2, ax3) = plt.subplots(1,3)
        fig.set_figheight(6)
        fig.set_figwidth(18)
        ax1.imshow(ventricle_is, cmap =  "Greens")
        ax2.imshow(ventricle_ap[::-1,:], cmap = 'gray')
        ax3.imshow(ventricle_lr[::-1,:], cmap = 'gray')
        plt.show()

        fig,ax = plt.subplots(figsize = (5,5))
        fig.set_figheight(6)
        fig.set_figwidth(18)
        ax.imshow(np.sum(full_brain, axis = 0), cmap = 'gray')
        plt.show()
    
    return ventricles, full_brain

##### Data setup

###### Before running the pipeline, make a copy of config.template.json, rename it to config.json, and update the paths to point to your local dataset.

In [ ]:
# Load the configuration file
with open('config.json', 'r') as file:
    config = json.load(file)

# Assign variables dynamically
data_path = config['data_path']
axial_acpc_vol_aligned_name = config['axial_acpc_vol_aligned_name']
info_csv_path = config['info_csv_path_w_name']
scan_id_col_name = config['scan_id_col_name']
pat_id_col_name = config['pat_id_col_name']
ac_coordinates_col_name = config['ac_coordinates_col_name']
pc_coordinates_col_name = config['pc_coordinates_col_name']

output_folder = config['output_csv_write_folder']

print(f"Loading data from: {data_path}")

In [ ]:
info_df = pd.read_csv(info_csv_path)
info_df[scan_id_col_name] = info_df[scan_id_col_name].str.strip("'")

In [ ]:
accnum_list = info_df[scan_id_col_name].values

In [ ]:
len(accnum_list), len(info_df)

In [ ]:
#dataframe which will contain track of the computed CSF2BVR values for each scan
csf2brain_df = pd.DataFrame()

##### CSF2BVR pipeline

In [ ]:
for acc_num in accnum_list:

    pat = info_df[pat_id_col_name][info_df[scan_id_col_name]==acc_num].values[0]
    
    print(f'Patient is {pat} and acc_num is {acc_num}')
    study_path = os.path.join(data_path, acc_num, axial_acpc_vol_aligned_name) 

    #Read in axial volume
    ax_img = sitk.ReadImage(study_path)

    ax_img_brain = window_stack_sitk(rotate_image(ax_img), 50, 100)
    ax_img_brain_arr = sitk.GetArrayFromImage(ax_img_brain)
    
    ventricles, full_brain = ventricle_fullbrain_seg(ax_img_brain_arr)

    csf2brain = np.sum(ventricles)/np.sum(full_brain)

    
    csf2brain_df = pd.concat([csf2brain_df, pd.DataFrame({pat_id_col_name:[pat],scan_id_col_name:["'" + acc_num + "'"],
                                                     'csf2brainR':[csf2brain]})],
                                                     ignore_index = True)

        
        

In [ ]:
len(csf2brain_df)

In [ ]:
#Examine histogram of computed values
plt.hist(csf2brain_df['csf2brainR'])
plt.xticks([x for x in np.arange(0, 0.3, 0.05)])
plt.show()

In [ ]:
#Check for any extreme values and inspect the corresponding outputs above for computational errors
min_csf2bvr_lim = 0.01
max_csf2bvr_lim = 0.3

csf2brain_df[(csf2brain_df['csf2brainR'] <= min_csf2bvr_lim) | (csf2brain_df['csf2brainR'] >= max_csf2bvr_lim)]

In [ ]:
csf2brain_df.to_csv(os.path.join(output_folder, 'csf2brain.csv'))